# Model Evaluation & Comparison

Compare trained YOLOX models across detection tasks (fire, helmet, shoes, fall, phone).

**Sections:**
1. Setup & configuration
2. Load evaluation results
3. Per-class AP comparison table
4. Side-by-side confusion matrices
5. Overlaid PR curves
6. Acceptance criteria check (Phase 1 targets)

In [ ]:
import sys
from pathlib import Path

# Add pipeline root to path
PIPELINE_ROOT = Path.cwd().parent if Path.cwd().name == "04_evaluation" else Path.cwd()
sys.path.insert(0, str(PIPELINE_ROOT))

import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
from IPython.display import display, HTML

from utils.config import load_config
from core.p04_evaluation.visualization import plot_confusion_matrix, plot_pr_curve

%matplotlib inline
plt.rcParams["figure.dpi"] = 120
plt.rcParams["font.size"] = 10

## 1. Configuration

Define the models to compare and their evaluation output directories. Update the paths below to point to your actual evaluation outputs (generated by `evaluate.py`).

In [ ]:
# --- Update these paths to match your evaluation runs ---
EVAL_BASE = PIPELINE_ROOT / "outputs" / "eval"

MODEL_CONFIGS = {
    "fire":          {"eval_dir": EVAL_BASE / "fire",          "config": PIPELINE_ROOT / "configs" / "data" / "fire.yaml"},
    "helmet":        {"eval_dir": EVAL_BASE / "helmet",        "config": PIPELINE_ROOT / "configs" / "data" / "helmet.yaml"},
    "shoes":         {"eval_dir": EVAL_BASE / "shoes",         "config": PIPELINE_ROOT / "configs" / "data" / "shoes.yaml"},
    "fall_classify": {"eval_dir": EVAL_BASE / "fall_classify", "config": PIPELINE_ROOT / "configs" / "data" / "fall_classify.yaml"},
    "phone":         {"eval_dir": EVAL_BASE / "phone",         "config": PIPELINE_ROOT / "configs" / "data" / "phone.yaml"},
}

# Phase 1 acceptance criteria (from project specs)
ACCEPTANCE_CRITERIA = {
    "fire":          {"precision": 0.90, "recall": 0.88, "max_fp_rate": 0.03, "max_fn_rate": 0.05},
    "helmet":        {"precision": 0.94, "recall": 0.92, "max_fp_rate": 0.02, "max_fn_rate": 0.03},
    "shoes":         {"precision": 0.88, "recall": 0.85, "max_fp_rate": 0.04, "max_fn_rate": 0.05},
    "fall_classify": {"precision": 0.90, "recall": 0.88, "max_fp_rate": 0.03, "max_fn_rate": 0.02},
    "phone":         {"precision": 0.85, "recall": 0.82, "max_fp_rate": 0.05, "max_fn_rate": 0.06},
}

print("Configured models:", list(MODEL_CONFIGS.keys()))

## 2. Load Evaluation Results

Load `metrics.json` from each model's evaluation directory.

In [ ]:
results = {}

for model_name, cfg in MODEL_CONFIGS.items():
    metrics_path = cfg["eval_dir"] / "metrics.json"
    if not metrics_path.exists():
        print(f"  [SKIP] {model_name}: {metrics_path} not found")
        continue
    with open(metrics_path) as f:
        results[model_name] = json.load(f)
    print(f"  [OK]   {model_name}: mAP={results[model_name]['mAP']:.4f}")

if not results:
    print("\nNo evaluation results found. Run evaluate.py first for each model.")
else:
    print(f"\nLoaded results for {len(results)} model(s).")

## 3. Per-Class AP Comparison Table

Summary table showing mAP and per-class AP for all evaluated models.

In [ ]:
if results:
    # Build comparison table
    header = f"{'Model':<16s} {'mAP':>8s}"
    # Collect all class names across models
    all_classes = set()
    for r in results.values():
        all_classes.update(r.get("per_class", {}).keys())
    all_classes = sorted(all_classes)
    for cls in all_classes:
        header += f" {cls:>12s}"
    
    print("=" * len(header))
    print(header)
    print("-" * len(header))
    
    for model_name, r in sorted(results.items()):
        row = f"{model_name:<16s} {r['mAP']:>8.4f}"
        per_class = r.get("per_class", {})
        for cls in all_classes:
            if cls in per_class:
                row += f" {per_class[cls]['ap']:>12.4f}"
            else:
                row += f" {'--':>12s}"
        print(row)
    
    print("=" * len(header))
else:
    print("No results to display.")

## 4. Side-by-Side Confusion Matrices

Display confusion matrices from each model's evaluation output.

In [ ]:
if results:
    available_cms = []
    for model_name, cfg in MODEL_CONFIGS.items():
        cm_path = cfg["eval_dir"] / "confusion_matrix.png"
        if cm_path.exists():
            available_cms.append((model_name, cm_path))
    
    if available_cms:
        n = len(available_cms)
        ncols = min(n, 3)
        nrows = (n + ncols - 1) // ncols
        fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 5 * nrows))
        axes = np.asarray(axes).ravel() if n > 1 else [axes]
        
        for i, (model_name, cm_path) in enumerate(available_cms):
            img = plt.imread(str(cm_path))
            axes[i].imshow(img)
            axes[i].set_title(model_name, fontsize=12, fontweight="bold")
            axes[i].axis("off")
        
        for j in range(i + 1, len(axes)):
            axes[j].axis("off")
        
        fig.suptitle("Confusion Matrices", fontsize=14, fontweight="bold", y=1.02)
        fig.tight_layout()
        plt.show()
    else:
        print("No confusion matrix images found. Run evaluate.py first.")
else:
    print("No results loaded.")

## 5. Overlaid PR Curves

Plot precision-recall curves for all models/classes on the same axes for comparison.

In [ ]:
if results:
    # Collect all PR curve images per model
    pr_data = {}
    for model_name, cfg in MODEL_CONFIGS.items():
        eval_dir = cfg["eval_dir"]
        if not eval_dir.exists():
            continue
        pr_files = sorted(eval_dir.glob("pr_curve_*.png"))
        if pr_files:
            pr_data[model_name] = pr_files
    
    if pr_data:
        for model_name, pr_files in pr_data.items():
            n = len(pr_files)
            fig, axes = plt.subplots(1, n, figsize=(6 * n, 4.5))
            if n == 1:
                axes = [axes]
            
            for i, pr_path in enumerate(pr_files):
                img = plt.imread(str(pr_path))
                axes[i].imshow(img)
                cls_name = pr_path.stem.replace("pr_curve_", "")
                axes[i].set_title(f"{model_name} / {cls_name}", fontsize=10)
                axes[i].axis("off")
            
            fig.suptitle(f"PR Curves: {model_name}", fontsize=12, fontweight="bold")
            fig.tight_layout()
            plt.show()
    else:
        print("No PR curve images found. Run evaluate.py first.")

    # Also create a summary mAP bar chart across models
    if len(results) > 1:
        fig, ax = plt.subplots(figsize=(8, 4))
        names = sorted(results.keys())
        maps = [results[n]["mAP"] for n in names]
        bars = ax.bar(range(len(names)), maps, color=plt.cm.Set2(np.linspace(0, 1, len(names))))
        ax.set_xticks(range(len(names)))
        ax.set_xticklabels(names, rotation=30, ha="right")
        ax.set_ylabel("mAP@0.5")
        ax.set_title("mAP Comparison Across Models")
        ax.set_ylim(0, 1.0)
        for bar, v in zip(bars, maps):
            ax.text(bar.get_x() + bar.get_width() / 2, v + 0.01, f"{v:.3f}",
                    ha="center", va="bottom", fontsize=9)
        ax.grid(axis="y", alpha=0.3)
        fig.tight_layout()
        plt.show()
else:
    print("No results loaded.")

## 6. Acceptance Criteria Check

Compare each model's metrics against the Phase 1 acceptance targets defined in `01_plan/00_phase1_requirements_spec.md`.

In [ ]:
if results:
    print("=" * 85)
    print(f"  {'Model':<16s} {'Metric':<12s} {'Achieved':>10s} {'Target':>10s} {'Status':>8s}")
    print("=" * 85)
    
    all_pass = True
    for model_name in sorted(results.keys()):
        r = results[model_name]
        criteria = ACCEPTANCE_CRITERIA.get(model_name)
        if criteria is None:
            print(f"  {model_name:<16s} {'--':<12s} {'N/A':>10s} {'N/A':>10s} {'SKIP':>8s}")
            continue
        
        # Compute average precision and recall across classes
        per_class = r.get("per_class", {})
        if per_class:
            avg_prec = np.mean([c["precision"] for c in per_class.values()])
            avg_rec = np.mean([c["recall"] for c in per_class.values()])
        else:
            avg_prec = 0.0
            avg_rec = 0.0
        
        # Precision check
        p_ok = avg_prec >= criteria["precision"]
        status = "PASS" if p_ok else "FAIL"
        if not p_ok:
            all_pass = False
        print(f"  {model_name:<16s} {'Precision':<12s} {avg_prec:>10.4f} {criteria['precision']:>10.2f} {status:>8s}")
        
        # Recall check
        r_ok = avg_rec >= criteria["recall"]
        status = "PASS" if r_ok else "FAIL"
        if not r_ok:
            all_pass = False
        print(f"  {'':<16s} {'Recall':<12s} {avg_rec:>10.4f} {criteria['recall']:>10.2f} {status:>8s}")
        
        # mAP (informational)
        print(f"  {'':<16s} {'mAP@0.5':<12s} {r['mAP']:>10.4f} {'--':>10s} {'INFO':>8s}")
        print("-" * 85)
    
    print("=" * 85)
    overall = "ALL PASS" if all_pass else "SOME FAIL"
    print(f"\n  Overall status: {overall}")
    if not all_pass:
        print("  Action: Review failing models. Consider more data, longer training, or architecture escalation.")
else:
    print("No results loaded. Run evaluate.py for each model first.")